# 📘 Notebook 18: Reinforcement Learning in Practice

### Course: *From Random Moves to Winning Agents: Reinforcement Learning with Games*
**Instructor:** Ioannis Tsioulis

*Module E: Beyond a Single Agent · Notebook 3 of 3 in this module · (18 of 18 overall)*

---

## 🎯 Learning objectives

By the end of this notebook you will be able to:

- Explain why a **single training run** proves little in reinforcement learning, with a measurement
- Report results properly: **several seeds**, an average, and an honest statement of the uncertainty
- Estimate how many runs a comparison needs before its conclusion can be trusted
- Recognise **reward hacking** and repair a reward that invites it
- Place every algorithm of the course on one **map**, and choose among them for a new problem
- Plan and carry out a small reinforcement learning **project** of your own

> **Prerequisites:** the rest of the course. This notebook collects its lessons, and Notebook 07 in particular is used again.
>
> 🔭 **Why this notebook matters:** an algorithm that you can write is worth little if you cannot tell whether it worked. Almost every notebook of this course contained a moment where a single run would have led to the wrong conclusion. This last notebook is about not being fooled, by the agent or by yourself.

> 📦 **Libraries used in this notebook.** NumPy, matplotlib and **Gymnasium** (only for the base class of the Snake environment). The main experiment trains 40 small agents and takes under a minute.

In [ ]:
%pip install -q "gymnasium>=1.3"

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import gymnasium as gym

## 1. A field in which results wobble

Look back over the course. Again and again the same thing happened:

| Notebook | What we saw |
|---|---|
| 01 | In a single game the greedy agent beat the exploring one. Averaged over 500 games it was clearly worse. |
| 07 | Five runs of the same Snake learner without exploration scored between 2 and 10. |
| 08 | Three runs of naive deep Q-learning ended at 38, 17 and 116 steps. |
| 10 | Refinements of DQN made no measurable difference on CartPole, and on Flappy Bird the runs of different variants overlapped. |
| 13 | A2C on Lunar Lander: 180 with one seed, −33 with the next. |
| 17 | An agent that drew a hundred games against a perfect player could still be beaten. |

None of these were accidents. They follow from what reinforcement learning is.

- **The agent creates its own data.** A lucky or unlucky early episode changes the policy, which changes all later data. Small differences grow.
- **Randomness enters everywhere:** the initial weights, the exploration, the environment, the minibatches.
- **Performance is not a smooth function of training time.** An agent can be good at step 100,000 and poor at step 110,000.

The remedy is not a better algorithm. It is a better **experiment**.

## 2. One run is an anecdote

To have something concrete to measure, we bring back the tabular Snake learner of Notebook 07. It trains in a fraction of a second, so we can afford many runs. The next cell is the environment and the learner from there, unchanged. Run it and move on.

In [ ]:
class SnakeEnv(gym.Env):
    """Snake on a square grid. The agent sees 11 yes/no features, not the whole board."""

    metadata = {"render_modes": ["rgb_array"], "render_fps": 10}
    DIRECTIONS = [(0, -1), (1, 0), (0, 1), (-1, 0)]        # up, right, down, left, as (dx, dy)

    def __init__(self, size=10, render_mode="rgb_array"):
        self.size = size
        self.render_mode = render_mode
        self.observation_space = gym.spaces.MultiBinary(11)
        self.action_space = gym.spaces.Discrete(3)          # 0 straight on, 1 turn right, 2 turn left

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)                            # prepares self.np_random
        middle = self.size // 2
        self.snake = [(middle, middle), (middle - 1, middle), (middle - 2, middle)]     # head first
        self.direction = 1                                  # moving right
        self.score = 0
        self.steps_since_food = 0
        self._place_food()
        return self._observation(), {"score": self.score}

    def _place_food(self):
        free = [(x, y) for x in range(self.size) for y in range(self.size) if (x, y) not in self.snake]
        self.food = free[int(self.np_random.integers(len(free)))]

    def _cell_ahead(self, direction):
        dx, dy = self.DIRECTIONS[direction]
        return (self.snake[0][0] + dx, self.snake[0][1] + dy)

    def _is_deadly(self, cell):
        x, y = cell
        return not (0 <= x < self.size and 0 <= y < self.size) or cell in self.snake

    def _observation(self):
        d, head, food = self.direction, self.snake[0], self.food
        features = [
            self._is_deadly(self._cell_ahead(d)),               # danger straight ahead
            self._is_deadly(self._cell_ahead((d + 1) % 4)),     # danger on my right
            self._is_deadly(self._cell_ahead((d - 1) % 4)),     # danger on my left
            d == 0, d == 1, d == 2, d == 3,                     # moving up, right, down, left
            food[0] < head[0], food[0] > head[0],               # food is to the left, to the right
            food[1] < head[1], food[1] > head[1],               # food is above, below
        ]
        return np.array(features, dtype=np.int8)

    def step(self, action):
        self.direction = (self.direction + (0, 1, -1)[action]) % 4
        new_head = self._cell_ahead(self.direction)
        self.steps_since_food += 1
        if self._is_deadly(new_head):                           # hit a wall or itself
            cause = "body" if new_head in self.snake else "wall"
            return self._observation(), -10.0, True, False, {"score": self.score, "death": cause}
        self.snake.insert(0, new_head)
        reward, terminated = 0.0, False
        if new_head == self.food:
            self.score += 1
            self.steps_since_food = 0
            reward = 10.0
            if len(self.snake) == self.size ** 2:               # the board is full: a perfect game
                terminated = True
            else:
                self._place_food()
        else:
            self.snake.pop()                                    # no food: the tail moves on
        truncated = self.steps_since_food >= 2 * self.size ** 2     # going round in circles
        return self._observation(), reward, terminated, truncated, {"score": self.score}

    def render(self, cell=24):
        image = np.full((self.size * cell, self.size * cell, 3), 25, dtype=np.uint8)

        def paint(position, colour, margin=2):
            x, y = position
            image[y * cell + margin:(y + 1) * cell - margin, x * cell + margin:(x + 1) * cell - margin] = colour

        paint(self.food, (220, 60, 60))
        for part in self.snake[1:]:
            paint(part, (70, 170, 90))
        paint(self.snake[0], (160, 235, 130), margin=1)
        return image


POWERS = 2 ** np.arange(11)


def row(observation):
    return int(observation @ POWERS)


def q_learning(env, games, alpha=0.1, gamma=0.9, explore_games=300, seed=0):
    """Tabular Q-learning on Snake (Notebook 07). Returns the Q-table."""
    rng = np.random.default_rng(seed)
    Q = np.zeros((2048, 3))
    for game in range(games):
        epsilon = max(0.01, 1 - game / explore_games)
        observation, info = env.reset(seed=seed * 100000 + game)
        state = row(observation)
        terminated = truncated = False
        while not (terminated or truncated):
            if rng.random() < epsilon:
                action = int(rng.integers(3))
            else:
                action = int(rng.choice(np.flatnonzero(Q[state] == Q[state].max())))
            observation, reward, terminated, truncated, info = env.step(action)
            next_state = row(observation)
            Q[state, action] += alpha * (reward + gamma * (0.0 if terminated else Q[next_state].max()) - Q[state, action])
            state = next_state
    return Q


def evaluate(env, Q, games=50):
    """Average score of the greedy policy over a fixed set of games."""
    scores = []
    for game in range(games):
        observation, info = env.reset(seed=50000 + game)
        terminated = truncated = False
        while not (terminated or truncated):
            observation, reward, terminated, truncated, info = env.step(int(Q[row(observation)].argmax()))
        scores.append(info["score"])
    return float(np.mean(scores))

### The question
Somebody proposes to train with a step size of `alpha = 0.5` in place of `0.1`, arguing that larger steps should learn faster. Is that an improvement?

The quick way to find out: train one agent with each setting and compare.

In [ ]:
env = SnakeEnv()

for seed in (3, 4, 5):
    score_a = evaluate(env, q_learning(env, 400, alpha=0.1, seed=seed))
    score_b = evaluate(env, q_learning(env, 400, alpha=0.5, seed=seed))
    verdict = "alpha = 0.5 is better" if score_b > score_a else "alpha = 0.1 is better"
    print(f"seed {seed}: alpha 0.1 scores {score_a:5.1f},  alpha 0.5 scores {score_b:5.1f}   ->  {verdict}")

Three people who each ran this comparison once, with a different seed, walk away with different conclusions, and each of them has a number to show for it.

### The proper way
Train **twenty** agents with each setting. This takes well under a minute.

In [ ]:
SEEDS = range(20)
results = {
    "alpha = 0.1": np.array([evaluate(env, q_learning(env, 400, alpha=0.1, seed=seed)) for seed in SEEDS]),
    "alpha = 0.5": np.array([evaluate(env, q_learning(env, 400, alpha=0.5, seed=seed)) for seed in SEEDS]),
}

plt.figure(figsize=(8, 3.5))
for position, (name, scores) in enumerate(results.items()):
    plt.scatter(scores, [position] * len(scores), s=45, alpha=0.7)
    plt.scatter([scores.mean()], [position], marker="|", s=900, color="black")
plt.yticks(range(len(results)), results.keys()); plt.ylim(-0.6, 1.6)
plt.title("Final score of 20 training runs per setting (the bar marks the average)")
plt.xlabel("average score of the trained agent"); plt.grid(True, axis="x")
plt.show()

for name, scores in results.items():
    print(f"{name}: average {scores.mean():5.1f},  standard deviation {scores.std(ddof=1):4.1f},  "
          f"worst run {scores.min():5.1f},  best run {scores.max():5.1f}")

Each dot is one complete training run. Three things can be read from the picture that no single run could show.

- The two clouds **overlap** widely. The best run of the worse setting beats most runs of the better one.
- The **averages** differ: the smaller step size is ahead.
- The **spread** differs too. One of the settings produces an occasional very poor run. For somebody who can train only once, reliability may matter more than the average.

How likely was the quick comparison to mislead? We have the data to say: take every possible pair of one run from each setting.

In [ ]:
a, b = results["alpha = 0.1"], results["alpha = 0.5"]
wrong = np.mean(b[None, :] > a[:, None])
print(f"alpha = 0.1 is better on average by {a.mean() - b.mean():.1f} points.")
print(f"Yet in {100 * wrong:.0f}% of all pairings, the single run with alpha = 0.5 scored higher.")

A comparison based on one run of each would have pointed the wrong way in a large share of cases. A great deal of what is believed about reinforcement learning methods rests on comparisons of this kind, which is one reason to read claims in this field with care.

## 3. Saying how sure you are

### Intuition first
An opinion poll that asks 20 people is not worthless, but nobody would report its result to a tenth of a percent. Twenty training runs are a poll of all the runs you might have made. The average of the poll is your best guess, and the honest question is how far the truth may lie from it.

### Formal definition
A simple and general tool is the **bootstrap**. Pretend that the 20 results are the whole population. Draw 20 of them at random **with replacement**, compute the average, and repeat many times. The middle 95% of those averages is a **95% confidence interval**: a range that would contain the true average in about 95 of 100 such experiments.

In [ ]:
def bootstrap_interval(values, repeats=10000, seed=0):
    """95% confidence interval for the mean, by resampling."""
    rng = np.random.default_rng(seed)
    means = rng.choice(values, size=(repeats, len(values)), replace=True).mean(axis=1)
    return np.percentile(means, [2.5, 97.5])


for name, scores in results.items():
    low, high = bootstrap_interval(scores)
    print(f"{name}: average {scores.mean():5.1f},  95% confidence interval from {low:5.1f} to {high:5.1f}")

rng = np.random.default_rng(0)
differences = (rng.choice(a, size=(10000, len(a))).mean(axis=1) - rng.choice(b, size=(10000, len(b))).mean(axis=1))
low, high = np.percentile(differences, [2.5, 97.5])
print(f"\ndifference of the averages: {a.mean() - b.mean():.1f},  95% confidence interval from {low:.1f} to {high:.1f}")

The last line is the one to look at. If the interval for the **difference** lies entirely above zero, the data support the claim that the first setting is better. If it includes zero, they do not, however the two averages happen to compare.

Read your interval with that in mind. In our run its lower end lies almost exactly at zero: even twenty training runs per setting only just make the case. For a difference that looked obvious in the averages, that is a sobering result.

### How many runs are enough?
There is no fixed number, but we can ask our own data. Take the 20 results per setting as the truth, and suppose we had run only $n$ seeds of each. How often would the comparison of the two averages have come out the other way round?

In [ ]:
rng = np.random.default_rng(1)
for n in (1, 3, 5, 10, 20):
    mean_a = rng.choice(a, size=(20000, n)).mean(axis=1)
    mean_b = rng.choice(b, size=(20000, n)).mean(axis=1)
    print(f"{n:>2} runs per setting: alpha = 0.5 has the higher average in {100 * np.mean(mean_b > mean_a):4.1f}% of experiments")

The risk of a wrong conclusion falls as the number of runs grows, and it falls slowly. The numbers depend on the problem: a large difference between two methods shows up with a few runs, and a small one may need dozens. Three to five seeds, the common practice, are enough only for large effects.

### Rules for reporting

- **Never report the best run.** Report all of them, or the average with its spread.
- **Decide the number of seeds before you look at the results.** Adding runs until the picture looks right is a way of fooling yourself.
- **Evaluate separately from training**, without exploration, on a fixed set of test episodes.
- **Tune and test on different seeds.** A setting chosen because it did well on seeds 0 to 4 will look better on those seeds than it really is.
- **Report what did not work.** The tables in Notebooks 10, 13 and 14 include the unflattering rows on purpose.

## 4. Agents do what you reward

Notebook 07 gave the snake +1 for every step it survived, and got a snake that ran in circles. Such failures have a name, **reward hacking** or **specification gaming**, and they are not limited to toys. A well-known example was published by OpenAI in 2016: an agent trained on a boat-racing video game, rewarded for hitting targets along the course, learned to circle for ever through a small group of targets that reappeared, catching fire and crashing as it went, and scored more points than human players who finished the race.

The mechanism can be reproduced in a few lines and solved exactly with the value iteration of Notebook 03. A robot walks along a corridor of six cells to a goal at the right-hand end, worth +10. To "encourage progress", the designer puts a bonus of +2 on the third cell.

In [ ]:
def solve_corridor(bonus_once, gamma=0.95, bonus=2.0, goal=10.0, cells=6):
    """Value iteration for the corridor. A state is (cell, has the bonus been collected?)."""
    V = np.zeros((cells, 2))

    def result(cell, collected, action):                  # action 0: left, 1: right
        nxt = min(cells - 1, max(0, cell + (1 if action else -1)))
        reward, taken = 0.0, collected
        if nxt == 2 and not (bonus_once and collected):
            reward, taken = bonus, 1
        if nxt == cells - 1:
            return reward + goal, None                    # the goal ends the episode
        return reward, (nxt, taken)

    def q(cell, collected, action):
        reward, nxt = result(cell, collected, action)
        return reward + (0.0 if nxt is None else gamma * V[nxt])

    for sweep in range(500):
        for cell in range(cells - 1):
            for collected in (0, 1):
                V[cell, collected] = max(q(cell, collected, action) for action in (0, 1))

    # follow the greedy policy from the left-hand end for 12 steps
    cell, collected, path = 0, 0, [0]
    for step in range(12):
        action = int(np.argmax([q(cell, collected, action) for action in (0, 1)]))
        reward, nxt = result(cell, collected, action)
        if nxt is None:
            path.append(cells - 1)
            break
        cell, collected = nxt
        path.append(cell)
    return path


print("bonus every time the cell is entered: the optimal path is", solve_corridor(bonus_once=False))
print("bonus only the first time:            the optimal path is", solve_corridor(bonus_once=True))

With a bonus that can be collected again and again, the **optimal** policy never reaches the goal: it steps on and off the bonus cell for ever. This is not a failure of learning. Value iteration is exact, and it is telling us that we asked for the wrong thing. Paying the bonus only once restores the intended behaviour.

Three habits protect against this.

- **Reward outcomes, not methods** (Notebook 06). Every extra term in a reward is a new thing the agent may maximise in place of what you wanted.
- **Ask what a lazy genius would do.** Before training, look for the cheapest way to collect reward. If it is not the behaviour you want, the agent will probably find it.
- **Watch the agent.** A score says how much reward was collected, not how. Every notebook in this course recorded videos for this reason.

## 5. The course on one page

| Algorithm | Notebook | Learns | Works from | Actions | Uses old data |
|---|---|---|---|---|---|
| Value iteration | 03 | values | a known model | discrete, few states | does not need data |
| Monte Carlo control | 04 | action values | whole episodes | discrete | no |
| SARSA | 05 | action values | single steps | discrete | no (on-policy) |
| Q-learning | 05, 07 | action values | single steps | discrete | yes (off-policy) |
| DQN and its refinements | 09, 10, 11 | action values (network) | single steps, replay buffer | discrete | yes |
| REINFORCE | 12 | a policy | whole episodes | discrete or continuous | no |
| A2C | 13 | a policy and a critic | short rollouts | discrete or continuous | no |
| PPO | 14, 15 | a policy and a critic | rollouts, several passes | discrete or continuous | only within one rollout |
| SAC, TD3 | 15 | a policy and critics | single steps, replay buffer | continuous | yes |
| Dyna-Q | 16 | action values and a model | real and imagined steps | discrete | yes, through the model |
| MCTS | 16 | nothing: it plans | a known model | discrete | does not need data |

### Choosing for a new problem

| If... | then start with... |
|---|---|
| the states can be listed (thousands, not millions) | tabular Q-learning. It is simple and can be inspected. |
| the model is known and the problem is a board game | tree search, possibly with self-play (Notebooks 16 and 17) |
| the actions are discrete and experience is cheap | PPO |
| the actions are discrete and experience is expensive | DQN with its refinements |
| the actions are continuous and experience is cheap | PPO with a Gaussian policy |
| the actions are continuous and experience is expensive | SAC |
| there is an opponent | decide first what the agent trains against (Notebook 17) |

Whatever the choice, the order of work that this course has followed is worth keeping:

1. **Define the environment** and write down the five decisions of Notebook 06.
2. **Measure baselines**: a random agent and the simplest rule you can think of.
3. **Start with the simplest algorithm** that could work, on the smallest version of the problem.
4. **Watch it**, with videos, and plot learning curves.
5. **Use several seeds** from the start, and evaluate separately from training.
6. Only then make the problem harder or the algorithm cleverer, **one change at a time**.

## 6. Where reinforcement learning stands

**Where it has worked.** Board and video games are the best-known successes: backgammon in the 1990s, Atari games (Notebook 11), Go, chess and shogi (Notebook 16), and large multiplayer games such as Dota 2 and StarCraft II. Outside games it is used to control legged robots and robot hands, usually after training in simulation, and in products such as recommendation systems. It is also one of the tools used to adjust large language models, by rewarding answers that human raters prefer.

**Where it struggles.** The weaknesses are the ones you have met.

- **It needs a great deal of experience.** Hundreds of thousands of steps for a lunar lander, over a billion configured for Pokémon (Notebook 11). That is acceptable in a simulator and rarely in the physical world.
- **Simulators are not reality.** An agent trained in simulation meets conditions it has not seen, like the lander in the wind (Notebook 09).
- **The reward has to be written down**, and a reward that says exactly what we mean is hard to write (Section 4).
- **Results are fragile**: sensitive to settings, seeds and details of implementation (Section 2).
- **Exploration of rare rewards** remains an open problem (Notebooks 05 and 11).

Reinforcement learning is the right tool when decisions follow one another, their effects are delayed, and trial and error is affordable. When a good solution can be written by hand, as with the tactician of Notebook 17, or when labelled examples exist, simpler methods are usually better.

## 7. Your own project

The best way to consolidate the course is to build something from the first decision to the final plot. A suitable project is small enough to train in minutes and rich enough to surprise you.

**The brief**

1. **Choose or invent a small game.** Write it as a Gymnasium environment with `reset`, `step` and `render`, and check it with `check_env` (Notebook 06).
2. **State your five decisions**: actions, observation, reward, end of episode, rendering.
3. **Measure two baselines**: a random agent, and a rule you write by hand.
4. **Train an agent** with an algorithm of your choice, and justify the choice with the table of Section 5.
5. **Evaluate with at least five seeds**: report the average, the spread and a confidence interval, next to the baselines.
6. **Record a video** of the trained agent, and one of an early, untrained agent.
7. **Change one thing** (the observation, the reward, or a setting), predict the effect, and measure it.
8. **Write one page**: what you built, what worked, what did not, and what surprised you.

**Some ideas**

| Game | A natural first algorithm | What makes it interesting |
|---|---|---|
| Catch with several balls, or with a ball that speeds up | DQN on features, then on pixels (Notebook 11) | the observation: what must the agent see? |
| A maze with keys and doors | tabular Q-learning, Dyna-Q | rare rewards and exploration |
| A simple platform jumper or a "dinosaur" runner | DQN or PPO | timing, and exploration that suits the game (Notebook 10) |
| Balancing or steering a vehicle with a throttle | PPO or SAC (Notebook 15) | continuous actions |
| Connect Four against the MCTS of Notebook 16 | self-play Q-learning, or tuning the search | measuring strength against a known opponent |
| A richer battle game: more types, more monsters, status effects | Q-learning against a pool (Notebook 17) | the observation and the choice of opponents |

### Where to go from here

- **Richard Sutton and Andrew Barto, *Reinforcement Learning: An Introduction*** (second edition, 2018). The standard textbook, freely available online from the authors. Notebooks 01 to 05 and 16 follow its first half closely.
- **OpenAI's *Spinning Up in Deep RL***: a free online guide to the policy-gradient family of Module D, with clear derivations.
- **The documentation of Gymnasium and Stable-Baselines3**: the two libraries used in this course, each with many more environments and algorithms than we touched.
- **CleanRL**: single-file reference implementations of the main algorithms, good for reading alongside your own.

---
## ✏️ Exercises

The exercises use `results`, `bootstrap_interval`, `q_learning`, `evaluate`, `env` and `solve_corridor` from above.

### Exercise 1 (A confidence interval by hand)
Five training runs gave the scores 12, 15, 9, 16 and 13. Compute their average. Then draw three bootstrap samples "by hand" with `np.random.default_rng(0).choice(scores, 5)`, compute the average of each, and compare with the interval that `bootstrap_interval` reports. Why is the interval so wide?

In [ ]:
# Your solution here:


<details><summary>💡 Show solution</summary>

```python
scores = np.array([12, 15, 9, 16, 13])
print("average:", scores.mean())

rng = np.random.default_rng(0)
for i in range(3):
    sample = rng.choice(scores, 5)
    print(f"bootstrap sample {i + 1}: {sample}, average {sample.mean():.1f}")

low, high = bootstrap_interval(scores)
print(f"95% confidence interval: from {low:.1f} to {high:.1f}")
```

*With five runs the average is known only roughly: the interval spans several points. A claim such as "method A scores 13.0 and method B 13.8, so B is better" cannot be supported by five runs with this much scatter.*
</details>

### Exercise 2 (Another comparison)
Notebook 07, Exercise 4, compared discount factors on Snake with three seeds. Do it properly: train 20 agents with `gamma=0.5` (400 games each) and compare with the `alpha = 0.1` results above, which used `gamma=0.9`. Report both averages, the confidence interval of the difference, and the share of single-run pairings in which each setting wins. Do the data support a difference?

In [ ]:
# Your solution here:


<details><summary>💡 Show solution</summary>

```python
gamma_low = np.array([evaluate(env, q_learning(env, 400, gamma=0.5, seed=seed)) for seed in SEEDS])
gamma_high = results["alpha = 0.1"]

rng = np.random.default_rng(0)
differences = (rng.choice(gamma_low, size=(10000, 20)).mean(axis=1) - rng.choice(gamma_high, size=(10000, 20)).mean(axis=1))
low, high = np.percentile(differences, [2.5, 97.5])

print(f"gamma = 0.9: average {gamma_high.mean():.1f},  standard deviation {gamma_high.std(ddof=1):.1f}")
print(f"gamma = 0.5: average {gamma_low.mean():.1f},  standard deviation {gamma_low.std(ddof=1):.1f}")
print(f"difference (0.5 minus 0.9): {gamma_low.mean() - gamma_high.mean():+.1f},  95% confidence interval from {low:+.1f} to {high:+.1f}")
print(f"single-run pairings won by gamma = 0.5: {100 * np.mean(gamma_low[None, :] > gamma_high[:, None]):.0f}%")
```

*Look at whether the interval of the difference contains zero. If it does, twenty runs per setting have not shown a difference in the average, whatever three runs suggested in Notebook 07. The standard deviations may still differ, and a setting whose runs scatter less is the safer choice even when the averages agree.*
</details>

### Exercise 3 (How good is the best of several?)
A common shortcut is to train several agents and keep the best. Using the 20 results for `alpha = 0.1`, estimate by resampling the expected score of "the best of n runs" for n = 1, 3, 5 and 10. Then explain why reporting that number as the performance of the **method** would be misleading, and what it legitimately tells you.

In [ ]:
# Your solution here:


<details><summary>💡 Show solution</summary>

```python
rng = np.random.default_rng(0)
scores = results["alpha = 0.1"]
for n in (1, 3, 5, 10):
    best = rng.choice(scores, size=(20000, n)).max(axis=1).mean()
    print(f"best of {n:>2} runs: expected score {best:.1f}   (average of a single run: {scores.mean():.1f})")
```

*The best of ten runs scores clearly above the average of one. As a description of the method that is flattering and false: somebody who trains once will get the average, not the maximum. As a recipe it is legitimate: if what you need is one good agent, train several and keep the best, provided you then evaluate the chosen agent on fresh test episodes, since it was selected for doing well on the old ones.*
</details>

### Exercise 4 (Another reward that goes wrong)
In the corridor of Section 4 the designer tries a different idea: no bonus cell, but a penalty of −1 for every step, "so that the robot hurries", and the goal is now worth only +3. The corridor is extended to 10 cells, and there is a pit just left of the start that ends the episode with a reward of 0. Without writing code, compute the return of walking to the goal (9 steps, gamma = 1) and the return of stepping into the pit at once. What will the optimal policy do? Check your arithmetic in Python.

In [ ]:
# Your solution here:


<details><summary>💡 Show solution</summary>

```python
to_the_goal = 9 * (-1) + 3
into_the_pit = -1 + 0
print(f"walking to the goal:    {to_the_goal}")
print(f"stepping into the pit:  {into_the_pit}")
print("the optimal policy:", "walk to the goal" if to_the_goal > into_the_pit else "end the episode at once")
```

*Nine steps at −1 cost more than the goal is worth, so the best thing the robot can do is to end the episode as fast as possible. The same thing happened in Notebook 03, where the policy improved from random values walked into pits, and it is a classic trap: with a penalty per step, any way of ending the episode early becomes attractive. The sizes of the rewards relative to each other matter as much as their signs.*
</details>

### Exercise 5 (Classify the agents)
For each of these agents of the course, say whether it is (a) value-based or policy-based, (b) on-policy or off-policy, and (c) model-free or using a model: the Blackjack agent of Notebook 04, the snake of Notebook 07, the lander of Notebook 09, the lander of Notebook 14, the pendulum agent trained with SAC in Notebook 15, the Connect Four player of Notebook 16.

<details><summary>💡 Show solution</summary>

```python
# Notebook 04, Blackjack:      value-based, on-policy (it evaluates the epsilon-greedy policy it follows), model-free
# Notebook 07, Snake:          value-based, off-policy (Q-learning), model-free
# Notebook 09, lander (DQN):   value-based, off-policy, model-free
# Notebook 14, lander (PPO):   policy-based with a critic, on-policy, model-free
# Notebook 15, SAC:            policy-based with critics, off-policy, model-free
# Notebook 16, Connect Four:   neither value- nor policy-based in the learned sense: it plans with a known model
```

*The three questions are independent of each other, which is why the table of Section 5 needs several columns. Most confusion about reinforcement learning algorithms disappears once each of them is placed on these three axes.*
</details>

### Exercise 6 (A project plan (a little harder))
Choose one of the project ideas of Section 7, or one of your own, and write the plan before any code: the five design decisions, the two baselines, the algorithm and why, the number of seeds, what you will plot, and one prediction that the experiment could prove wrong. Then carry it out.

<details><summary>💡 Show solution</summary>

```python
# There is no single solution. A good plan can be checked against these questions:
#   - Could a person play the game with only the observation you give the agent?
#   - What is the cheapest way to collect your reward, and is it what you want?
#   - How will an episode end if the agent finds a safe loop?
#   - What score do the random agent and the hand-written rule reach?
#   - How many seeds, decided in advance? What exactly will be on each axis of each plot?
#   - Which result would make you conclude that your idea did not work?
```

*The last question is the most important one. An experiment that cannot fail cannot teach you anything.*
</details>

## 🔑 Key takeaways

- In reinforcement learning the agent creates its own data, so **runs with different seeds differ widely**. One run is an anecdote.
- Compare methods on **many runs**: report the average, the spread and a **confidence interval**, never the best run.
- Whether a difference is real depends on its size relative to the scatter. Small effects need **many seeds**.
- Agents maximise the reward **as written**. Look for the cheapest way to collect it before the agent does, and **watch** what it learns.
- Every algorithm of the course has a place on three axes: **value or policy**, **on- or off-policy**, **with or without a model**.
- Work from the simple to the complex: **environment, baselines, simplest algorithm, several seeds, one change at a time**.

### 🎓 You have completed the course!

You started with a row of slot machines and a single question: use what I know, or try something new? Since then you have:

- solved small worlds exactly with the **Bellman equations**,
- learned a card game from wins and losses, and a **snake game** of your own making with a table,
- replaced the table by a **neural network**, repaired what that broke, and landed a spacecraft,
- taught an agent to play from **raw pixels**,
- learned policies directly, up to **PPO** and **continuous actions**,
- made agents **think ahead** with tree search and improve by **playing against themselves**,
- and learned to doubt a result until several seeds agree.

The algorithms will keep changing. The habits will not: build the smallest thing that could work, measure it against a baseline, watch what the agent does, and trust averages over anecdotes.

Now build something of your own. 🎉

---
*End of the course.*

---
*© Ioannis Tsioulis. *From Random Moves to Winning Agents: Reinforcement Learning with Games*. Prepared for educational use.*